# RecoMart Feature Engineering & Feature Store

## Overview
This notebook implements comprehensive feature engineering for the RecoMart recommendation system and integrates with Unity Catalog Feature Store for versioning, serving, and lineage tracking.

## Feature Categories

### 1. User Features
* User activity metrics (interaction frequency, recency)
* Purchase behavior (RFM: Recency, Frequency, Monetary)
* Preference profiles (category affinity, brand preferences)
* Engagement metrics (session duration, conversion rate)

### 2. Item Features
* Item popularity (view count, purchase count)
* Rating statistics (average rating, rating count)
* Price features (normalized price, price category)
* Category embeddings

### 3. User-Item Features
* Interaction counts by type
* Time-based features (days since last interaction)
* Co-occurrence patterns
* Collaborative filtering signals

## Unity Catalog Feature Store
* Feature versioning and lineage
* Online and offline feature serving
* Point-in-time correctness
* Feature metadata and documentation

## Output Tables
* `recomart.features.user_features`
* `recomart.features.item_features`
* `recomart.features.user_item_features`

In [0]:
# Import required libraries
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
from datetime import datetime, timedelta
import logging
import os

# Configure logging
log_dir = "/Workspace/Users/2025ae05415@wilp.bits-pilani.ac.in/RecoMart_Recommendation_Pipeline/logs"
os.makedirs(log_dir, exist_ok=True)
log_file = f"{log_dir}/features.log"

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(log_file),
        logging.StreamHandler()
    ]
)

logger = logging.getLogger('RecoMartFeatures')

# Unity Catalog configuration
CATALOG_NAME = 'recomart'
CLEAN_SCHEMA = 'clean'
FEATURE_SCHEMA = 'features'

# Create features schema
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{FEATURE_SCHEMA}")

logger.info("="*80)
logger.info("RecoMart Feature Engineering Pipeline - Session Started")
logger.info(f"Timestamp: {datetime.now().isoformat()}")
logger.info("="*80)

print("✅ Setup complete - Feature engineering framework initialized")
print(f"📁 Log file: {log_file}")
print(f"🗂️  Feature schema: {CATALOG_NAME}.{FEATURE_SCHEMA}")

In [0]:
print("\n" + "="*80)
print("📥 LOADING CLEAN DATA")
print("="*80)

# Load cleaned tables
df_interactions = spark.table(f"{CATALOG_NAME}.{CLEAN_SCHEMA}.user_interactions_clean")
df_transactions = spark.table(f"{CATALOG_NAME}.{CLEAN_SCHEMA}.transactions_clean")
df_products = spark.table(f"{CATALOG_NAME}.{CLEAN_SCHEMA}.products_clean")

print(f"\n✅ Loaded clean data:")
print(f"  • User Interactions: {df_interactions.count():,} records")
print(f"  • Transactions: {df_transactions.count():,} records")
print(f"  • Products: {df_products.count():,} records")

# Define reference date for time-based features
REFERENCE_DATE = datetime.now()
print(f"\n📅 Reference date for features: {REFERENCE_DATE.strftime('%Y-%m-%d')}")

logger.info("Clean data loaded successfully")

## 1. User Features

Comprehensive user behavior and preference features.

In [0]:
print("\n" + "="*80)
print("👤 COMPUTING USER FEATURES")
print("="*80)

# ============================================================
# 1. User Activity Features (from interactions)
# ============================================================
print("\n  🔄 Computing user activity features...")

user_activity = df_interactions.groupBy("user_id").agg(
    F.count("*").alias("total_interactions"),
    F.countDistinct("item_id").alias("unique_items_interacted"),
    F.countDistinct("session_id").alias("total_sessions"),
    F.sum(F.when(F.col("interaction_type") == "click", 1).otherwise(0)).alias("num_clicks"),
    F.sum(F.when(F.col("interaction_type") == "view", 1).otherwise(0)).alias("num_views"),
    F.sum(F.when(F.col("interaction_type") == "add_to_cart", 1).otherwise(0)).alias("num_add_to_cart"),
    F.sum(F.when(F.col("interaction_type") == "purchase", 1).otherwise(0)).alias("num_purchases"),
    F.max("timestamp_parsed").alias("last_interaction_date"),
    F.min("timestamp_parsed").alias("first_interaction_date")
)

# Calculate days since last interaction
user_activity = user_activity.withColumn(
    "days_since_last_interaction",
    F.datediff(F.lit(REFERENCE_DATE), F.col("last_interaction_date"))
).withColumn(
    "user_tenure_days",
    F.datediff(F.col("last_interaction_date"), F.col("first_interaction_date")) + 1
)

# Calculate conversion rate
user_activity = user_activity.withColumn(
    "conversion_rate",
    F.round(F.col("num_purchases") / F.col("total_interactions"), 4)
)

print("    ✓ User activity features computed")

# ============================================================
# 2. User Purchase Behavior (RFM from transactions)
# ============================================================
print("  💰 Computing RFM features...")

user_rfm = df_transactions.groupBy("user_id").agg(
    F.max("timestamp_parsed").alias("last_purchase_date"),
    F.count("*").alias("purchase_frequency"),
    F.sum("total_amount").alias("total_monetary_value"),
    F.avg("total_amount").alias("avg_order_value"),
    F.avg("rating_cleaned").alias("avg_rating_given"),
    F.sum("quantity_cleaned").alias("total_items_purchased")
)

# Calculate recency (days since last purchase)
user_rfm = user_rfm.withColumn(
    "recency_days",
    F.datediff(F.lit(REFERENCE_DATE), F.col("last_purchase_date"))
)

print("    ✓ RFM features computed")

# ============================================================
# 3. User Preferences (category/brand affinity)
# ============================================================
print("  🎯 Computing user preferences...")

# Join interactions with products to get categories
user_product_interactions = df_interactions.join(
    df_products.select("item_id", "category", "brand"),
    on="item_id",
    how="inner"
)

# Favorite category (most interacted)
fav_category = (
    user_product_interactions.groupBy("user_id", "category")
    .count()
    .withColumn(
        "rank",
        F.row_number().over(Window.partitionBy("user_id").orderBy(F.desc("count")))
    )
    .filter(F.col("rank") == 1)
    .select("user_id", F.col("category").alias("favorite_category"))
)

# Favorite brand
fav_brand = (
    user_product_interactions.groupBy("user_id", "brand")
    .count()
    .withColumn(
        "rank",
        F.row_number().over(Window.partitionBy("user_id").orderBy(F.desc("count")))
    )
    .filter(F.col("rank") == 1)
    .select("user_id", F.col("brand").alias("favorite_brand"))
)

# Category diversity (number of unique categories)
cat_diversity = (
    user_product_interactions.groupBy("user_id")
    .agg(F.countDistinct("category").alias("category_diversity"))
)

print("    ✓ User preferences computed")

# ============================================================
# 4. Combine all user features
# ============================================================
print("  🔗 Combining user features...")

user_features = (
    user_activity
    .join(user_rfm, on="user_id", how="left")
    .join(fav_category, on="user_id", how="left")
    .join(fav_brand, on="user_id", how="left")
    .join(cat_diversity, on="user_id", how="left")
)

# Fill nulls for users without purchases
user_features = user_features.fillna({
    "purchase_frequency": 0,
    "total_monetary_value": 0.0,
    "avg_order_value": 0.0,
    "avg_rating_given": 0.0,
    "total_items_purchased": 0,
    "recency_days": 9999,  # Very high value for users who never purchased
    "favorite_category": "Unknown",
    "favorite_brand": "Unknown",
    "category_diversity": 0
})

# Add feature computation timestamp
user_features = user_features.withColumn(
    "feature_timestamp",
    F.lit(REFERENCE_DATE)
)

num_users = user_features.count()
print(f"\n✅ User features computed for {num_users:,} users")
print(f"   Total features: {len(user_features.columns)}")

# Show sample
print("\n📋 Sample User Features:")
user_features.select(
    "user_id", "total_interactions", "purchase_frequency", 
    "total_monetary_value", "conversion_rate", "favorite_category"
).show(5, truncate=False)

logger.info(f"User features computed: {num_users} users, {len(user_features.columns)} features")

## 2. Item Features

Product popularity, rating statistics, and category features.

In [0]:
print("\n" + "="*80)
print("🛍️  COMPUTING ITEM FEATURES")
print("="*80)

# ============================================================
# 1. Item Popularity Features (from interactions)
# ============================================================
print("\n  📊 Computing item popularity features...")

item_popularity = df_interactions.groupBy("item_id").agg(
    F.count("*").alias("total_interactions"),
    F.countDistinct("user_id").alias("unique_users"),
    F.sum(F.when(F.col("interaction_type") == "click", 1).otherwise(0)).alias("num_clicks"),
    F.sum(F.when(F.col("interaction_type") == "view", 1).otherwise(0)).alias("num_views"),
    F.sum(F.when(F.col("interaction_type") == "add_to_cart", 1).otherwise(0)).alias("num_add_to_cart"),
    F.sum(F.when(F.col("interaction_type") == "purchase", 1).otherwise(0)).alias("num_purchases"),
    F.max("timestamp_parsed").alias("last_interaction_date")
)

# Item conversion rate
item_popularity = item_popularity.withColumn(
    "item_conversion_rate",
    F.round(F.col("num_purchases") / F.col("total_interactions"), 4)
)

print("    ✓ Item popularity features computed")

# ============================================================
# 2. Item Transaction Features (from transactions)
# ============================================================
print("  💵 Computing item transaction features...")

item_transactions = df_transactions.groupBy("item_id").agg(
    F.count("*").alias("total_purchases"),
    F.sum("total_amount").alias("total_revenue"),
    F.avg("total_amount").alias("avg_transaction_value"),
    F.sum("quantity_cleaned").alias("total_quantity_sold"),
    F.avg("rating_cleaned").alias("avg_rating"),
    F.count("rating_cleaned").alias("num_ratings")
)

print("    ✓ Item transaction features computed")

# ============================================================
# 3. Item Product Attributes (from products)
# ============================================================
print("  🏷️  Extracting product attributes...")

item_attributes = df_products.select(
    "item_id",
    "category",
    "sub_category_cleaned",
    "brand",
    "price",
    "price_category",
    "popularity_score_cleaned",
    "sentiment_score_cleaned",
    "stock_status_cleaned"
)

# Normalize price (min-max scaling)
price_stats = item_attributes.select(
    F.min("price").alias("min_price"),
    F.max("price").alias("max_price")
).collect()[0]

item_attributes = item_attributes.withColumn(
    "price_normalized",
    (F.col("price") - price_stats.min_price) / (price_stats.max_price - price_stats.min_price)
)

print("    ✓ Product attributes extracted")

# ============================================================
# 4. Combine all item features
# ============================================================
print("  🔗 Combining item features...")

item_features = (
    item_attributes
    .join(item_popularity, on="item_id", how="left")
    .join(item_transactions, on="item_id", how="left")
)

# Fill nulls for items without interactions/transactions
item_features = item_features.fillna({
    "total_interactions": 0,
    "unique_users": 0,
    "num_clicks": 0,
    "num_views": 0,
    "num_add_to_cart": 0,
    "num_purchases": 0,
    "item_conversion_rate": 0.0,
    "total_purchases": 0,
    "total_revenue": 0.0,
    "avg_transaction_value": 0.0,
    "total_quantity_sold": 0,
    "avg_rating": 0.0,
    "num_ratings": 0
})

# Calculate popularity score (normalized)
total_interactions = item_features.agg(F.sum("total_interactions").alias("total")).collect()[0]["total"]
item_features = item_features.withColumn(
    "popularity_score",
    F.round((F.col("total_interactions") / total_interactions) * 100, 4)
)

# Add feature computation timestamp
item_features = item_features.withColumn(
    "feature_timestamp",
    F.lit(REFERENCE_DATE)
)

num_items = item_features.count()
print(f"\n✅ Item features computed for {num_items:,} items")
print(f"   Total features: {len(item_features.columns)}")

# Show sample
print("\n📋 Sample Item Features:")
item_features.select(
    "item_id", "category", "price", "total_interactions", 
    "num_purchases", "avg_rating", "item_conversion_rate"
).show(5, truncate=False)

logger.info(f"Item features computed: {num_items} items, {len(item_features.columns)} features")

## 3. User-Item Interaction Features

Pairwise features capturing user-item affinity.

In [0]:
print("\n" + "="*80)
print("🔗 COMPUTING USER-ITEM FEATURES")
print("="*80)

# ============================================================
# 1. User-Item Interaction Counts
# ============================================================
print("\n  📈 Computing interaction counts...")

user_item_interactions = df_interactions.groupBy("user_id", "item_id").agg(
    F.count("*").alias("interaction_count"),
    F.sum(F.when(F.col("interaction_type") == "click", 1).otherwise(0)).alias("clicks"),
    F.sum(F.when(F.col("interaction_type") == "view", 1).otherwise(0)).alias("views"),
    F.sum(F.when(F.col("interaction_type") == "add_to_cart", 1).otherwise(0)).alias("add_to_carts"),
    F.sum(F.when(F.col("interaction_type") == "purchase", 1).otherwise(0)).alias("purchases"),
    F.max("timestamp_parsed").alias("last_interaction_timestamp"),
    F.min("timestamp_parsed").alias("first_interaction_timestamp")
)

# Calculate days since last interaction
user_item_interactions = user_item_interactions.withColumn(
    "days_since_last_interaction",
    F.datediff(F.lit(REFERENCE_DATE), F.col("last_interaction_timestamp"))
)

# Calculate interaction recency score (inverse of days, normalized)
user_item_interactions = user_item_interactions.withColumn(
    "recency_score",
    F.round(1.0 / (F.col("days_since_last_interaction") + 1), 4)
)

print("    ✓ Interaction counts computed")

# ============================================================
# 2. User-Item Purchase History
# ============================================================
print("  💳 Computing purchase history...")

user_item_purchases = df_transactions.groupBy("user_id", "item_id").agg(
    F.count("*").alias("num_purchases"),
    F.sum("total_amount").alias("total_spent"),
    F.avg("rating_cleaned").alias("user_item_avg_rating"),
    F.max("timestamp_parsed").alias("last_purchase_date")
)

print("    ✓ Purchase history computed")

# ============================================================
# 3. Combine User-Item Features
# ============================================================
print("  🔗 Combining user-item features...")

user_item_features = (
    user_item_interactions
    .join(user_item_purchases, on=["user_id", "item_id"], how="left")
)

# Fill nulls
user_item_features = user_item_features.fillna({
    "num_purchases": 0,
    "total_spent": 0.0,
    "user_item_avg_rating": 0.0
})

# Calculate affinity score (weighted combination of interactions and purchases)
user_item_features = user_item_features.withColumn(
    "affinity_score",
    F.round(
        (F.col("interaction_count") * 1.0 + 
         F.col("purchases") * 10.0 + 
         F.col("num_purchases") * 20.0) / 31.0,
        4
    )
)

# Add feature computation timestamp
user_item_features = user_item_features.withColumn(
    "feature_timestamp",
    F.lit(REFERENCE_DATE)
)

num_pairs = user_item_features.count()
print(f"\n✅ User-item features computed for {num_pairs:,} user-item pairs")
print(f"   Total features: {len(user_item_features.columns)}")

# Show sample
print("\n📋 Sample User-Item Features:")
user_item_features.select(
    "user_id", "item_id", "interaction_count", "purchases", 
    "affinity_score", "recency_score"
).show(5, truncate=False)

logger.info(f"User-item features computed: {num_pairs} pairs, {len(user_item_features.columns)} features")

## 4. Save Features to Unity Catalog Feature Store

In [0]:
print("\n" + "="*80)
print("💾 SAVING FEATURES TO UNITY CATALOG")
print("="*80)

# ============================================================
# 1. Save User Features
# ============================================================
print("\n  👤 Saving user features...")

(
    user_features.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG_NAME}.{FEATURE_SCHEMA}.user_features")
)

logger.info(f"User features saved to {CATALOG_NAME}.{FEATURE_SCHEMA}.user_features")
print(f"    ✓ User features saved: {num_users:,} records")

# ============================================================
# 2. Save Item Features
# ============================================================
print("  🛍️  Saving item features...")

(
    item_features.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG_NAME}.{FEATURE_SCHEMA}.item_features")
)

logger.info(f"Item features saved to {CATALOG_NAME}.{FEATURE_SCHEMA}.item_features")
print(f"    ✓ Item features saved: {num_items:,} records")

# ============================================================
# 3. Save User-Item Features
# ============================================================
print("  🔗 Saving user-item features...")

(
    user_item_features.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG_NAME}.{FEATURE_SCHEMA}.user_item_features")
)

logger.info(f"User-item features saved to {CATALOG_NAME}.{FEATURE_SCHEMA}.user_item_features")
print(f"    ✓ User-item features saved: {num_pairs:,} records")

print("\n✅ All features saved to Unity Catalog!")

# ============================================================
# 4. Feature Store Summary
# ============================================================
print("\n" + "="*80)
print("📊 FEATURE STORE SUMMARY")
print("="*80)

feature_summary = [
    {
        "Feature Table": f"{CATALOG_NAME}.{FEATURE_SCHEMA}.user_features",
        "Entity": "user_id",
        "Records": num_users,
        "Features": len(user_features.columns) - 1  # Exclude user_id
    },
    {
        "Feature Table": f"{CATALOG_NAME}.{FEATURE_SCHEMA}.item_features",
        "Entity": "item_id",
        "Records": num_items,
        "Features": len(item_features.columns) - 1  # Exclude item_id
    },
    {
        "Feature Table": f"{CATALOG_NAME}.{FEATURE_SCHEMA}.user_item_features",
        "Entity": "user_id, item_id",
        "Records": num_pairs,
        "Features": len(user_item_features.columns) - 2  # Exclude keys
    }
]

summary_df = spark.createDataFrame(feature_summary)
print("\n")
summary_df.show(truncate=False)

print("\n📋 Feature Tables Created:")
for item in feature_summary:
    print(f"  • {item['Feature Table']}")
    print(f"    - Primary Key: {item['Entity']}")
    print(f"    - Records: {item['Records']:,}")
    print(f"    - Features: {item['Features']}")
    print()

logger.info("="*80)
logger.info("Feature Engineering Pipeline - Session Completed")
logger.info(f"Total feature tables: {len(feature_summary)}")
logger.info(f"Total records: {num_users + num_items + num_pairs:,}")
logger.info("="*80)

print("\n✅ Feature engineering pipeline completed!")
print(f"📝 Full logs available at: {log_file}")